# BLM6106 — LLM-Guided Arithmetic Coding on Enwik8
**Project:** Approach 1 — GPT-2 as a Probability Model for Arithmetic Coding  
**Dataset:** Enwik8 (Hutter Prize benchmark, offset=0)  
**Models:** GPT-2 Small (117M) · GPT-2 Medium (345M) · GPT-2 XL (1.5B)  
**Reference:** Delétang et al., *Language Models are Compression Algorithms*, 2023

---

## Experiment plan

| Step | Run | Purpose |
|---|---|---|
| 1 | `test_edge_cases.py` | Verify arithmetic coder + CDF scaler (19 tests) |
| 2 | gpt2, 1MB | Main benchmark — full-scale BPC for the report |
| 3 | gpt2, 50KB `--full-encode` | Scaling curve point 1 + lossless verification |
| 4 | gpt2-medium, 50KB `--full-encode` | Scaling curve point 2 |
| 5 | gpt2-xl, 50KB `--full-encode` | Scaling curve point 3 |
| 6 | Results summary | Consolidated table + three-point scaling curve |

`--full-encode` reports both theoretical BPC (cross-entropy) and actual BPC (arithmetic coding),
and confirms losslessness — so Steps 3-5 each produce a complete result in one command.

**Note:** Step 2 takes ~78 min on T4. If you already have the result in `results/`, skip it.

**Estimated runtimes (Tesla T4):**
| Step | Time |
|---|---|
| Step 1 — edge cases | ~30 seconds |
| Step 2 — gpt2, 1MB | ~78 minutes |
| Step 3 — gpt2, 50KB + full encode | ~10 minutes |
| Step 4 — gpt2-medium, 50KB + full encode | ~20 minutes |
| Step 5 — gpt2-xl, 50KB + full encode | ~40 minutes |

---
## Step 0 — Setup
Run this cell first every time you open the notebook.

In [19]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")

import os
#home = '/content/drive/MyDrive/dcproject/lm_arithmetic_coding'
home = '/content/drive/MyDrive/dcproject/'
os.chdir(home)
print(f'Working directory: {os.getcwd()}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/dcproject


In [ ]:
# Install dependencies
!pip install psutil tqdm torch transformers --quiet

In [ ]:
!pip install torch transformers psutil tqdm --quiet
import torch, transformers
print(f'torch         : {torch.__version__}')
print(f'transformers  : {transformers.__version__}')
print(f'GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

torch         : 2.10.0+cu128
transformers  : 5.0.0
GPU           : Tesla T4
VRAM          : 15.6 GB


---
## Step 1 — Edge Case Verification

Tests the arithmetic coder and CDF scaling logic across 19 edge cases.  
Runs in ~30 seconds with no model download required.

Saves: `results/edge_case_report.txt`

In [ ]:
!python {home}/test_edge_cases.py

  BLM6106 — Edge Case Test Report

=== 0. bits_to_bytes / bytes_to_bits Roundtrip ===
  [PASS]  Empty bit list                                         
  [PASS]  Single 0                                               
  [PASS]  Single 1                                               
  [PASS]  Exactly 8 bits (0xAA)                                  
  [PASS]  8 x 1 (0xFF)                                           
  [PASS]  8 x 0 (0x00)                                           
  [PASS]  7 bits (non-multiple)                                  
  [PASS]  9 bits (non-multiple)                                  
  [PASS]  16 bits alternating                                    
  [PASS]  16 bits 1100 pattern                                   
  [PASS]  Random 1 bits                                          
  [PASS]  Random 7 bits                                          
  [PASS]  Random 8 bits                                          
  [PASS]  Random 9 bits                                 

---
## Step 2 — Main Benchmark: GPT-2 Small · 1MB

The primary result for the report. GPT-2 small on the full 1MB standard Enwik8 slice.  
**Skip this cell if `results/gpt2_1000k_start_*.json` already exists.**

Expected: ~1.14 BPC · ~7× compression ratio · ~78 min on T4

Saves: `results/gpt2_1000k_start_<timestamp>.json` + `_report.txt`

In [ ]:
!python {home}/evaluate.py --size 1000000 #1MB




[1/4] Preparing Enwik8 (size=1000KB, offset=0)...
[download] data/enwik8_1000k_start.txt already exists, skipping download.
      Loaded 1,000,000 bytes

[2/4] Running baseline compressors...
      Done in 3.8s
      Shannon entropy (lower bound): 5.0589 bpc
      Static Huffman: 5.0844 bpc
      Static Arithmetic Coding: 5.0589 bpc
      gzip -9: 2.8463 bpc
      bzip2 -9: 2.2506 bpc
      lzma preset=9: 2.3255 bpc

[3/4] Computing gpt2 cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2...
config.json: 100% 665/665 [00:00<00:00, 2.80MB/s]
model.safetensors: 100% 548M/548M [00:03<00:00, 177MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 655.26it/s, Materializing param=transformer.wte.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 455kB/s]
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 286272/286272 [1:18:28<00:00, 60.80tok/s, tok/s=61, ETA=0.0m]
      3.9843 bits/token  →  1.1406 bits/char
      Speed: 60.8 tok/s  |  Forward passes: 286,272 

In [ ]:
# BPC-only scan — 1MB at offset 300KB (prose region)
!python {home}/evaluate.py --size 1000000 --offset 300000 --model gpt2


[1/4] Preparing Enwik8 (size=1000KB, offset=300,000)...
[download] Using cached data/enwik8.zip
[download] Slicing 1,000,000 bytes from offset 300,000...
[download] Saved 1,000,000 bytes to data/enwik8_1000k_off300k.txt
      Loaded 1,000,000 bytes

[2/4] Running baseline compressors...
      Done in 3.6s
      Shannon entropy (lower bound): 5.0840 bpc
      Static Huffman: 5.1111 bpc
      Static Arithmetic Coding: 5.0840 bpc
      gzip -9: 2.8890 bpc
      bzip2 -9: 2.2939 bpc
      lzma preset=9: 2.3579 bpc

[3/4] Computing gpt2 cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2...
Loading weights: 100% 148/148 [00:00<00:00, 855.52it/s, Materializing param=transformer.wte.weight]
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 289158/289158 [1:05:28<00:00, 73.61tok/s, tok/s=74, ETA=0.0m]
      4.0428 bits/token  →  1.1690 bits/char
      Speed: 73.6 tok/s  |  Forward passes: 289,158  |  Mem Δ: 1047 MB

[4/4] Skipped full encode/decode (use --full-encode

---
## Step 3 — GPT-2 Small · 50KB · Full Encode/Decode

Runs theoretical BPC (cross-entropy) **and** actual arithmetic encode/decode in one command.  
The `--full-encode` flag encodes the first 5,000 bytes of the loaded 50KB slice and verifies losslessness.

Expected: ~1.12 BPC theoretical · ~0.88 BPC actual · ✓ LOSSLESS · ~10 min on T4

Saves: `results/gpt2_50k_start_<timestamp>.json` + `_report.txt`

In [ ]:
# GPT-2 Small — full encode/decode (~5 min)
!python {home}/evaluate.py --full-encode --size 50000 --model gpt2



[1/4] Preparing Enwik8 (size=50KB, offset=0)...
[download] data/enwik8_50k_start.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.4s
      Shannon entropy (lower bound): 4.8997 bpc
      Static Huffman: 4.9185 bpc
      Static Arithmetic Coding: 4.9045 bpc
      gzip -9: 3.0131 bpc
      bzip2 -9: 2.6960 bpc
      lzma preset=9: 2.8352 bpc

[3/4] Computing gpt2 cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2...
Loading weights: 100% 148/148 [00:02<00:00, 56.48it/s, Materializing param=transformer.wte.weight]
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 13447/13447 [08:26<00:00, 26.56tok/s, tok/s=27, ETA=0.0m]
      4.1568 bits/token  →  1.1179 bits/char
      Speed: 26.6 tok/s  |  Forward passes: 13,447  |  Mem Δ: 894 MB

[4/4] Full LM encode/decode on first 5000 bytes...
Encoding: 100% 2174/2174 [02:18<00:00, 15.67tok/s, tok/s=16]
[encode] 550 bytes (4,398 bits) in 138.7s — 16 to

In [ ]:
# GPT-2 Small — full encode/decode (~5 min)
!python {home}/evaluate.py --full-encode --size 50000 --offset 300000 --model gpt2


[1/4] Preparing Enwik8 (size=50KB, offset=300,000)...
[download] Using cached data/enwik8.zip
[download] Slicing 50,000 bytes from offset 300,000...
[download] Saved 50,000 bytes to data/enwik8_50k_off300k.txt
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.2s
      Shannon entropy (lower bound): 4.9793 bpc
      Static Huffman: 5.0041 bpc
      Static Arithmetic Coding: 4.9820 bpc
      gzip -9: 3.1278 bpc
      bzip2 -9: 2.7870 bpc
      lzma preset=9: 2.8678 bpc

[3/4] Computing gpt2 cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2...
Loading weights: 100% 148/148 [00:00<00:00, 751.50it/s, Materializing param=transformer.wte.weight] 
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 13677/13677 [03:03<00:00, 74.36tok/s, tok/s=74, ETA=0.0m]
      4.3184 bits/token  →  1.1813 bits/char
      Speed: 74.4 tok/s  |  Forward passes: 13,677  |  Mem Δ: 898 MB

[4/4] Full LM encode/decode on first 5000 bytes...
Encoding: 100% 185

---
## Step 4 — GPT-2 Medium · 50KB · Full Encode/Decode

GPT-2 Medium has 345M parameters — the middle point in the scaling curve.  
Adding this run gives three data points (117M → 345M → 1.5B), showing that  
BPC decreases monotonically and predictably with model size.  
Two data points establish a comparison; three establish a trend.

Expected: ~1.0 BPC theoretical · ~0.80 BPC actual · LOSSLESS · ~20 min on T4

Saves: `results/gpt2-medium_50k_start_<timestamp>.json` + `_report.txt`

In [ ]:
# GPT-2 Medium on 50KB (~20 min) — downloads ~1.4GB on first run
!python {home}/evaluate.py --full-encode --size 50000 --model gpt2-medium


[1/4] Preparing Enwik8 (size=50KB, offset=0)...
[download] data/enwik8_50k_start.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.8s
      Shannon entropy (lower bound): 4.8997 bpc
      Static Huffman: 4.9185 bpc
      Static Arithmetic Coding: 4.9045 bpc
      gzip -9: 3.0131 bpc
      bzip2 -9: 2.6960 bpc
      lzma preset=9: 2.8352 bpc

[3/4] Computing gpt2-medium cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2-medium...
config.json: 100% 718/718 [00:00<00:00, 2.03MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 95.0kB/s]
vocab.json: 1.04MB [00:00, 2.09MB/s]
merges.txt: 456kB [00:00, 9.42MB/s]
tokenizer.json: 1.36MB [00:00, 2.56MB/s]
model.safetensors: 100% 1.52G/1.52G [00:37<00:00, 40.9MB/s]
Loading weights: 100% 292/292 [00:01<00:00, 182.27it/s, Materializing param=transformer.wte.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 475kB/s]
[lm_compressor] Model loaded on cud

In [ ]:
# GPT-2 Medium (~20 min) — downloads ~1.4GB on first run
!python {home}/evaluate.py --full-encode --size 50000 --offset 300000 --model gpt2-medium


[1/4] Preparing Enwik8 (size=50KB, offset=300,000)...
[download] data/enwik8_50k_off300k.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.2s
      Shannon entropy (lower bound): 4.9793 bpc
      Static Huffman: 5.0041 bpc
      Static Arithmetic Coding: 4.9820 bpc
      gzip -9: 3.1278 bpc
      bzip2 -9: 2.7870 bpc
      lzma preset=9: 2.8678 bpc

[3/4] Computing gpt2-medium cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2-medium...
config.json: 100% 718/718 [00:00<00:00, 3.40MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 157kB/s]
vocab.json: 1.04MB [00:00, 5.94MB/s]
merges.txt: 456kB [00:00, 4.66MB/s]
tokenizer.json: 1.36MB [00:00, 18.0MB/s]
model.safetensors: 100% 1.52G/1.52G [00:07<00:00, 217MB/s]
Loading weights: 100% 292/292 [00:00<00:00, 339.54it/s, Materializing param=transformer.wte.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 539kB/s]
[lm_compressor] Model loaded 

---
## Step 5 — GPT-2 XL · 50KB · Full Encode/Decode

GPT-2 XL has 1.5B parameters — the largest and most capable model in the comparison.  
The model (~6GB) downloads automatically on first run.

Expected: ~0.87 BPC theoretical · ~0.69 BPC actual · LOSSLESS · ~40 min on T4

Saves: `results/gpt2-xl_50k_start_<timestamp>.json` + `_report.txt`

In [ ]:
# GPT-2 XL — full encode/decode (~20 min)
!python {home}/evaluate.py --full-encode --size 50000 --model gpt2-xl


[1/4] Preparing Enwik8 (size=50KB, offset=0)...
[download] data/enwik8_50k_start.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.4s
      Shannon entropy (lower bound): 4.8997 bpc
      Static Huffman: 4.9185 bpc
      Static Arithmetic Coding: 4.9045 bpc
      gzip -9: 3.0131 bpc
      bzip2 -9: 2.6960 bpc
      lzma preset=9: 2.8352 bpc

[3/4] Computing gpt2-xl cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2-xl...
Loading weights: 100% 580/580 [00:34<00:00, 16.99it/s, Materializing param=transformer.wte.weight]
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 13447/13447 [1:01:49<00:00,  3.63tok/s, tok/s=4, ETA=0.2m]
      3.2507 bits/token  →  0.8743 bits/char
      Speed: 3.6 tok/s  |  Forward passes: 13,447  |  Mem Δ: 853 MB

[4/4] Full LM encode/decode on first 5000 bytes...
Encoding: 100% 2174/2174 [09:43<00:00,  3.73tok/s, tok/s=4]
[encode] 429 bytes (3,430 bits) in 583.5s — 

In [ ]:
# GPT-2 XL — full encode/decode (~20 min)
!python {home}/evaluate.py --full-encode --size 50000 --offset 300000 --model gpt2-xl


[1/4] Preparing Enwik8 (size=50KB, offset=300,000)...
[download] data/enwik8_50k_off300k.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.2s
      Shannon entropy (lower bound): 4.9793 bpc
      Static Huffman: 5.0041 bpc
      Static Arithmetic Coding: 4.9820 bpc
      gzip -9: 3.1278 bpc
      bzip2 -9: 2.7870 bpc
      lzma preset=9: 2.8678 bpc

[3/4] Computing gpt2-xl cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2-xl...
config.json: 100% 689/689 [00:00<00:00, 3.12MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 160kB/s]
vocab.json: 1.04MB [00:00, 8.65MB/s]
merges.txt: 456kB [00:00, 12.2MB/s]
tokenizer.json: 1.36MB [00:00, 5.82MB/s]
model.safetensors: 100% 6.43G/6.43G [01:05<00:00, 97.7MB/s]
Loading weights: 100% 580/580 [00:21<00:00, 27.49it/s, Materializing param=transformer.wte.weight]
generation_config.json: 100% 124/124 [00:00<00:00, 729kB/s]
[lm_compressor] Model loaded on cuda.

---
## Step 6 — Window Size Ablation

Tests how much context length affects compression quality.  
The script reuses the existing window=512 result (Step 3) automatically  
and only runs the two missing sizes (128 and 256), each ~4 minutes.

Expected: BPC degrades as window shrinks — less context = worse predictions.  
The delta column shows the cost of reducing context relative to window=512.

Saves: `results/ablation_window_gpt2_50k_<timestamp>.json`

In [20]:
# Runs window=128 and window=256; reuses cached window=512 from Step 3
!python {home}/ablation_window.py


  Window = 128 tokens  |  model = gpt2  |  size = 50KB

[ablation] Running: /content/drive/MyDrive/dcproject/evaluate.py --model gpt2 --size 50000 --window 128 --offset 0

[1/4] Preparing Enwik8 (size=50KB, offset=0)...
[download] data/enwik8_50k_start.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.2s
      Shannon entropy (lower bound): 4.8997 bpc
      Static Huffman: 4.9185 bpc
      Static Arithmetic Coding: 4.9045 bpc
      gzip -9: 3.0131 bpc
      bzip2 -9: 2.6960 bpc
      lzma preset=9: 2.8352 bpc

[3/4] Computing gpt2 cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2...
Loading weights: 100% 148/148 [00:00<00:00, 596.80it/s, Materializing param=transformer.wte.weight]
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 13447/13447 [02:26<00:00, 91.87tok/s, tok/s=92, ETA=0.0m]
      4.6454 bits/token  →  1.2493 bits/char
      Speed: 91.9 tok/s  |  Forward passes: 13,447  |  Mem

In [21]:
# Window ablation — runs w=128 and w=256; reuses cached w=512 if offset matches
!python {home}/ablation_window.py --offset 300000


  Window = 128 tokens  |  model = gpt2  |  size = 50KB

[ablation] Running: /content/drive/MyDrive/dcproject/evaluate.py --model gpt2 --size 50000 --window 128 --offset 300000

[1/4] Preparing Enwik8 (size=50KB, offset=300,000)...
[download] data/enwik8_50k_off300k.txt already exists, skipping download.
      Loaded 50,000 bytes

[2/4] Running baseline compressors...
      Done in 0.2s
      Shannon entropy (lower bound): 4.9793 bpc
      Static Huffman: 5.0041 bpc
      Static Arithmetic Coding: 4.9820 bpc
      gzip -9: 3.1278 bpc
      bzip2 -9: 2.7870 bpc
      lzma preset=9: 2.8678 bpc

[3/4] Computing gpt2 cross-entropy BPC (with KV-cache)...
[lm_compressor] Loading gpt2...
Loading weights: 100% 148/148 [00:00<00:00, 789.69it/s, Materializing param=transformer.wte.weight] 
[lm_compressor] Model loaded on cuda.
Cross-entropy BPC: 100% 13677/13677 [02:24<00:00, 94.67tok/s, tok/s=95, ETA=0.0m]
      4.5424 bits/token  →  1.2425 bits/char
      Speed: 94.7 tok/s  |  Forward passes: 

---
## Step 7 — Results Summary

Reads all saved JSON files and prints a consolidated comparison table across all runs.

In [22]:
import json, glob, os

json_files = sorted(glob.glob(f'{home}/results/*.json'))
json_files = [f for f in json_files if 'gpt2' in f]
print(f'Found {len(json_files)} experiment result(s):\n')

for path in json_files:
    with open(path) as f:
        data = json.load(f)

    # Skip files without a results key
    if 'results' not in data:
        print(f'  [skip] {os.path.basename(path)} — no "results" key\n')
        continue

    # Skip if results is not a dict (e.g. ablation JSON stores a list)
    if not isinstance(data['results'], dict):
        print(f'  [skip] {os.path.basename(path)} — results is {type(data["results"]).__name__}, not dict\n')
        continue

    # Skip if entries don't have the bpc field
    sample_entry = next(iter(data['results'].values()), {})
    if 'bpc' not in sample_entry:
        print(f'  [skip] {os.path.basename(path)} — results lack "bpc" field\n')
        continue

    cfg    = data.get('config', {})
    run_id = data.get('run_id', os.path.basename(path))
    print(f"{'='*68}")
    print(f"  Run   : {run_id}")
    print(f"  Model : {cfg.get('model','?')}  |  Size: {cfg.get('size',0)//1000}KB")
    print(f"  {'Method':<44} {'BPC':>5}  {'Ratio':>6}  {'Saving':>7}")
    print(f"  {'-'*62}")
    for name, info in sorted(data['results'].items(), key=lambda x: x[1].get('bpc', 99)):
        bpc   = info.get('bpc',               0)
        ratio = info.get('compression_ratio', 0)
        save  = info.get('space_saving_pct',  0)
        print(f"  {name:<44} {bpc:>5.3f}  {ratio:>5.2f}x  {save:>6.1f}%")
    print()

Found 14 experiment result(s):

  [skip] ablation_window_gpt2_50k_off0k_w128_256_512_20260508_121332.json — results is list, not dict

  [skip] ablation_window_gpt2_50k_off300k_w128_256_512_20260508_121849.json — results is list, not dict

  Run   : gpt2-medium_50k_off300k_20260508_103201
  Model : gpt2-medium  |  Size: 50KB
  Method                                         BPC   Ratio   Saving
  --------------------------------------------------------------
  LLM-AC (gpt2-medium, theoretical)            1.039   7.70x    87.0%
  LLM-AC (gpt2-medium, actual 5KB)             1.062   7.53x    86.7%
  bzip2 -9                                     2.787   2.87x    65.2%
  lzma preset=9                                2.868   2.79x    64.2%
  gzip -9                                      3.128   2.56x    60.9%
  Shannon entropy (lower bound)                4.979   1.61x    37.8%
  Static Arithmetic Coding                     4.982   1.61x    37.7%
  Static Huffman                               5

In [24]:
# Model scaling comparison: LLM-AC vs best traditional compressor
print(f"Model Scaling — LLM-AC vs best traditional compressor")
print(f"{'Run':<45} {'LLM-AC BPC':>10}  {'Best trad.':>10}  {'Improvement':>12}")
print('-' * 82)

for path in json_files:
    with open(path) as f:
        data = json.load(f)

    # Skip files that don't have the expected schema
    if not isinstance(data.get('results'), dict):
        continue
    sample_entry = next(iter(data['results'].values()), {})
    if 'bpc' not in sample_entry:
        continue

    res = data['results']
    cfg = data.get('config', {})   # fallback to empty dict

    llm_key  = next((k for k in res if 'theoretical' in k), None)
    trad_key = min(
        (k for k in res if any(t in k for t in ['gzip', 'bzip2', 'lzma'])),
        key=lambda k: res[k]['bpc'], default=None
    )

    if llm_key and trad_key:
        llm_bpc  = res[llm_key]['bpc']
        trad_bpc = res[trad_key]['bpc']
        impr     = (trad_bpc - llm_bpc) / trad_bpc * 100
        label    = f"{cfg.get('model', '?')}  {cfg.get('size', 0) // 1000}KB"
        print(f"  {label:<43} {llm_bpc:>10.3f}  {trad_bpc:>10.3f}  {impr:>10.1f}% better")

Model Scaling — LLM-AC vs best traditional compressor
Run                                           LLM-AC BPC  Best trad.   Improvement
----------------------------------------------------------------------------------
  gpt2-medium  50KB                                1.039       2.787        62.7% better
  gpt2-medium  50KB                                0.986       2.696        63.4% better
  gpt2-xl  50KB                                    0.958       2.787        65.6% better
  gpt2-xl  50KB                                    0.874       2.696        67.6% better
  gpt2  1000KB                                     1.169       2.294        49.0% better
  gpt2  1000KB                                     1.141       2.251        49.3% better
  gpt2  50KB                                       1.181       2.787        57.6% better
  gpt2  50KB                                       1.243       2.787        55.4% better
  gpt2  50KB                                       1.200       2.787

---
## Notes for the Project Report

### Key claims
1. **Algorithm correctness:** All 19 edge case tests pass — lossless across 12 input types.
2. **AI integration depth:** GPT-2 *is* the probability model. The LLM forward pass directly produces the CDF fed to the arithmetic coder at every token position.
3. **Performance:** LLM-AC beats every traditional compressor. Gap widens with model size.
4. **Lossless:** `lossless: true` confirmed for both models in Steps 3 and 4.
5. **Reference:** Delétang et al. (2023) formally proves that LLM cross-entropy equals optimal compressed length — this experiment is a working instantiation of that theorem.

### Metrics required by the project sheet
| Metric | Where to find it |
|---|---|
| Compression ratio | Results table (every run) |
| Speed | `tokens_per_second` in each JSON |
| LLM token usage | `n_forward_passes` in each JSON |
| Lossless verification | `lossless: true` in Steps 3 & 4 JSONs |
| AI prompt history | `ai_log.md` |
| Bug identification + fix | Entries 2, 4, 5, 7 in `ai_log.md` |